In [1]:
import os
from PIL import Image
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Path to your dataset folder
dataset_path = 'Fruits'
processed_dataset_path = 'processed4'

# Create processed dataset directories if they don't exist
os.makedirs(os.path.join(processed_dataset_path, 'fresh'), exist_ok=True)
os.makedirs(os.path.join(processed_dataset_path, 'rotten'), exist_ok=True)

# Define the desired image size and format
desired_size = (224, 224)
desired_format = 'JPEG'

def preprocess_images(source_folder, target_folder):
    """Resize and convert images to the desired format."""
    for filename in os.listdir(source_folder):
        if filename.endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(source_folder, filename)
            with Image.open(img_path) as img:
                # Resize the image
                img = img.resize(desired_size)
                # Save the image in the desired format
                img.save(os.path.join(target_folder, filename), format=desired_format)

# Preprocess fresh and rotten images
preprocess_images(os.path.join(dataset_path, 'fresh'), os.path.join(processed_dataset_path, 'fresh'))
preprocess_images(os.path.join(dataset_path, 'rotten'), os.path.join(processed_dataset_path, 'rotten'))

# Check if images are present
fresh_images = os.listdir(os.path.join(processed_dataset_path, 'fresh'))
rotten_images = os.listdir(os.path.join(processed_dataset_path, 'rotten'))

if not fresh_images:
    print("No fresh images found in the directory.")
if not rotten_images:
    print("No rotten images found in the directory.")

# Preprocess data with ImageDataGenerator
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)  # Split data for training and validation
train_generator = train_datagen.flow_from_directory(
    processed_dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)
validation_generator = train_datagen.flow_from_directory(
    processed_dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

# Use MobileNetV2 for transfer learning
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze the base model layers

# Add custom classification layers
"""  
model = models.Sequential([
    base_model,
    layers.Ma(),
    layers.Dense(1, activation='sigmoid')  # Output layer for binary classification
])
"""
# Add custom classification layers
model = models.Sequential([
    base_model,                                   # Your base model (e.g., a pretrained model)      # Apply max pooling with a 2x2 pool size
    layers.GlobalAveragePooling2D(),              # Use Global Average Pooling to reduce dimensions
    layers.Dense(128, activation='relu'),         # First dense layer with 128 units
    layers.Dense(64, activation='relu'),          # Second dense layer with 64 units
    layers.Dense(32, activation='relu'),          # Third dense layer with 32 units
    layers.Dense(1, activation='sigmoid')         # Output layer for binary classification
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


# Train the model
model.fit(train_generator, validation_data=validation_generator, epochs=10)

# Save the model for future use
model.save('fruit_freshness_model3.h5')
print("model saved")

2024-10-19 22:03:05.383321: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-10-19 22:03:05.496808: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-19 22:03:05.539653: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-19 22:03:05.560038: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-19 22:03:05.635976: I tensorflow/core/platform/cpu_feature_guar

OSError: cannot write mode P as JPEG

In [5]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image

# Load the saved model
model = tf.keras.models.load_model('fruit_freshness_model2.h5')

def predict_freshness(img_path):
    # Load and preprocess the image
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img) / 255.0  # Rescale
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

    # Make prediction
    prediction = model.predict(img_array)
    print("fresness score",(1-prediction)*100)
    # Interpret the prediction
    if prediction[0] <= 0.6:
        return "Fresh"
    else:
        return "Rotten"

# Example usage
img_path = 'Fruits/rotten/39220258-rotten-banana.jpg'  # Path to the image you want to classify
result = predict_freshness(img_path)
print(f"The fruit is: {result}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 717ms/step
fresness score [[0.04556775]]
The fruit is: Rotten


In [3]:
import os
from PIL import Image
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Path to your dataset folder
dataset_path = 'Fruits'
processed_dataset_path = 'processed4'

# Create processed dataset directories if they don't exist
os.makedirs(os.path.join(processed_dataset_path, 'fresh'), exist_ok=True)
os.makedirs(os.path.join(processed_dataset_path, 'rotten'), exist_ok=True)

# Define the desired image size and format
desired_size = (224, 224)
desired_format = 'JPEG'

def preprocess_images(source_folder, target_folder):
    """Resize and convert images to the desired format."""
    for filename in os.listdir(source_folder):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):  # Make the file check case-insensitive
            img_path = os.path.join(source_folder, filename)
            with Image.open(img_path) as img:
                # Convert image to RGB (in case it's not)
                img = img.convert('RGB')
                # Resize the image
                img = img.resize(desired_size)
                # Save the image in the desired format
                img.save(os.path.join(target_folder, filename), format=desired_format)

# Preprocess fresh and rotten images
preprocess_images(os.path.join(dataset_path, 'fresh'), os.path.join(processed_dataset_path, 'fresh'))
preprocess_images(os.path.join(dataset_path, 'rotten'), os.path.join(processed_dataset_path, 'rotten'))

# Check if images are present
fresh_images = os.listdir(os.path.join(processed_dataset_path, 'fresh'))
rotten_images = os.listdir(os.path.join(processed_dataset_path, 'rotten'))

if not fresh_images:
    print("No fresh images found in the directory.")
if not rotten_images:
    print("No rotten images found in the directory.")

# Preprocess data with ImageDataGenerator
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)  # Split data for training and validation
train_generator = train_datagen.flow_from_directory(
    processed_dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)
validation_generator = train_datagen.flow_from_directory(
    processed_dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

# Use MobileNetV2 for transfer learning
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze the base model layers

# Add custom classification layers
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(train_generator, validation_data=validation_generator, epochs=20)

# Save the model for future use
model.save('fruit_freshness_model3.h5')
print("Model saved")


Found 49 images belonging to 2 classes.
Found 11 images belonging to 2 classes.
Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - accuracy: 0.5378 - loss: 0.7692 - val_accuracy: 0.8182 - val_loss: 0.6261
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 422ms/step - accuracy: 0.8307 - loss: 0.4998 - val_accuracy: 0.6364 - val_loss: 0.5631
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 406ms/step - accuracy: 0.9260 - loss: 0.3534 - val_accuracy: 0.8182 - val_loss: 0.4276
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 296ms/step - accuracy: 0.9760 - loss: 0.2431 - val_accuracy: 0.8182 - val_loss: 0.3248
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 425ms/step - accuracy: 0.9864 - loss: 0.1384 - val_accuracy: 0.9091 - val_loss: 0.2729
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 284ms/step - accuracy: 0.9760 - loss: 0.1093 - val_accuracy: 0.9091 - val_loss: 0.2242
Epoch 7/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 441ms/step - accuracy: 1.0000 - loss: 0.0553 - val_accuracy: 0.9091 - val_loss: 0.2225
Epoch 8/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 34

Model saved
